## IMPORT

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import sklearn
from datetime import datetime
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import joblib
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV

## MODÈLE 1

In [18]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

#/-------------------------------------------------------\#
#|------------- Tri et création de colonnes -------------|#
#\-------------------------------------------------------/#

# Récupère si l'article dans le panier est de l'alcool
df["is_alcohol"] = df["department"].str.contains("alcohol", case=False, na=False).astype(int)

# Création d'un nouveau dataset en récupérant la taille de la commande des clients en fonction de l'id de la commande
basket_size = df.groupby("order_id").size().reset_index(name="basket_size")

# Création d'un nouveau dataset en récupérant les données en fonction de l'id de la commande qui me permettra de trier plus tard
orders = df.groupby("order_id").agg(
    user_id=("user_id", "first"),
    order_number=("order_number", "first"),
    order_hour_of_day=("order_hour_of_day", "first"),
    order_dow=("order_dow", "first"),
    days_since_prior_order=("days_since_prior_order", "first"),
    alcohol_count=("is_alcohol", "sum"),
    reorder_count=("reordered", "sum"),
).reset_index()

# Ce merge là sert à trier une nouvelle fois en fusionnant avec le datasets basket_size en fonction de l'id de la commande
orders = orders.merge(basket_size, on="order_id")

# Récupère le nombre d'article d'alcool en fonction de la taille du panier
orders["alcohol_ratio"] = orders["alcohol_count"]/orders["basket_size"]

# Récupère le nombre d'article commendés plusieurs fois en fonction de la taille du panier
orders["reorder_ratio"]  = orders["reorder_count"]/orders["basket_size"]

# Création d'un nouveau dataset en récupérant les données en fonction de l'id du client qui me permettra de trier plus tard
user_stats = orders.groupby("user_id").agg(
    user_avg_basket=("basket_size", "mean"),
    user_std_basket=("basket_size", "std"),
    user_total_orders=("order_id", "count"),
    user_avg_alcohol_ratio=("alcohol_ratio", "mean"),
    user_avg_days_between=("days_since_prior_order", "mean"),
    user_max_basket=("basket_size", "max"),
    user_min_basket=("basket_size", "min"),
).reset_index()

# Remplacer les Na par des 0 dans la colonnes user_std_basket
user_stats["user_std_basket"] = user_stats["user_std_basket"].fillna(0)

# Ici je merge pour finir le tri
df2 = orders.merge(user_stats, on="user_id")

# Récupère si on est le week-end en récupérant à partir de la colonnes order_dow en prennant uniquement les valeurs entre 5 et 6 (donc Samedi et Dimanche)
df2["is_weekend"] = df2["order_dow"].isin([5, 6]).astype(int)
df2["time_of_day"] = pd.cut(
    df2["order_hour_of_day"],
    bins=[0, 6, 12, 17, 21],
    labels=["nuit", "matin", "apres_midi", "soir"],
    right=False
).astype(str)

#/--------------------------------------------------------------\#
#|------------------------ Entraînement ------------------------|#
#\--------------------------------------------------------------/#

# Ici je définis les colonnes que le modèle doit utiliser pour s'entraîner
feature_cols = [
    "order_hour_of_day", "order_dow",
    "is_weekend", "time_of_day",
    "days_since_prior_order", "order_number",
    "alcohol_count", "alcohol_ratio",
    "reorder_count", "reorder_ratio",
    "user_avg_basket", "user_std_basket", "user_total_orders",
    "user_avg_alcohol_ratio", "user_avg_days_between",
    "user_max_basket", "user_min_basket",
]

# Transforme les chaînes de charactère en True/False et définis mon X
X = pd.get_dummies(df2[feature_cols])

# Récupère le y
y = df2["basket_size"]

# Split les données entre tests et entraînements 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

# Définis le modèle utiliser et ses hyperparamètres
model = HistGradientBoostingRegressor(
    max_iter=500,                 
    learning_rate=0.05,
    max_leaf_nodes=127,            
    min_samples_leaf=50,                    
    random_state=42,
)

# Lance l'entraînements du modèle
model.fit(X_train, y_train)

#/--------------------------------------------------------------\#
#|------------------------- Évaluation -------------------------|#
#\--------------------------------------------------------------/#

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE :{mae}")
print(f"RMSE :{rmse}")
print(f"R2 :{r2}")



MAE :0.18295473386037947
RMSE :0.5055257857855556
R2 :0.9910240972893822
